# Amazon Top-K — A/B/C/D 통합 후보 생성 + LightGBM 재정렬 (Kaggle)

**데이터 누출을 막는 3가지 분할:** `inner`는 LightGBM 학습용(공통 val_train을 다시 분리, seed 7), `val`은 팀 공통 검증(seed 42, split_id `f0dfad2f4431a876`), `final`은 전체 train으로 학습하여 test 사용자를 예측한다. 모델마다 `user_id,item_id,score,rank` 형식의 **Top-100 압축 CSV(.csv.gz)**를 세 모드 모두 저장한다.

**우선순위:** 기존 D031(0.06082 val / 0.06705 Public)의 강력한 후보+LightGBM 구조를 기준으로, A047 **근사 재구현**(`A047_PROXY`), BM25/Text/Meta, FREEDOM, VBPR, EASE, RP3β, iALS를 합친다. A047의 원본 코드가 없으므로 정확 재현을 주장하지 않는다. B와 C의 LightGCN은 기존 D035에서 성능 저하가 보고되어 기본 실행에서 제외했다. `val_recall@20` 및 Public score 향상은 실험 전에 보장할 수 없다.

**리소스:** GPU 필수 권장. `PRESET="MAX"`는 최대 정확도 지향(3-seed VBPR/FREEDOM, 추가 3-seed 재정렬 탐색)으로 긴 GPU 시간이 소요된다. `BALANCED`는 1-seed/적은 epoch로 시험 실행한다. Kaggle에서 `Add Input`으로 대회 데이터(train/test), 가능하면 기존 팀 공통 validation dataset, `item_meta.jsonl`을 연결한다. `item_meta`가 없으면 텍스트/이미지 모델을 만들 수 없어 CF 중심 축소 실행한다. 인터넷이 차단되거나 이미지를 구할 수 없는 경우 TF-IDF/SVD 메타데이터 **대체 임베딩**을 사용하고 이는 원래 D031과 다른 모델로 기록한다. 

**실행:** 위에서 아래로 전체 실행 → `submission.csv`, `team_validation_scores.csv`, `TEAM_ALL_TOP100.zip`, 실행 설정 JSON 생성. 셀 도중 재시작 시 이미 저장된 Top100 파일은 건너뛴다. Kaggle `Save Version → Save & Run All`을 권장한다.

In [1]:
from pathlib import Path
import os,sys,json,subprocess,importlib.util, time
import numpy as np, pandas as pd
import torch

# === 사용자 설정 ===
PRESET = 'MAX'                # 'MAX' (3 seed, 긴 학습), 'BALANCED' (빠른 시험 실행)
FEATURE_MODE = 'auto'        # 'auto': 사전학습 임베딩 시도 -> 실패 시 proxy. 'proxy': 오프라인, 'real': 실제 CLIP 필수
DOWNLOAD_IMAGES = True      # 이미지 URL이 있고 Kaggle 인터넷이 가능할 때에만 자동 다운로드
MODEL_TOPN = 50 if PRESET == 'MAX' else 35
ALLOW_OPTIONAL_FAILURE = True
INCLUDE_PLAIN_CSV_IN_ZIP = True  # ZIP에 .csv.gz와 .csv를 모두 추가 (용량 증가)
RUN_EXTRA_RERANK = PRESET == 'MAX'  # 추가 이력-코사인 피처 및 3-seed LightGBM 탐색

assert PRESET in {'MAX','BALANCED'}
assert FEATURE_MODE in {'auto','proxy','real'}
if not torch.cuda.is_available():
    print('⚠ GPU 미탑재: Kaggle Notebook Settings → Accelerator → GPU를 권장합니다. 매우 오래 걸릴 수 있습니다.')
else:
    print('GPU:', torch.cuda.get_device_name(0))
for pkg in ['lightgbm','sentence_transformers','transformers']:
    if importlib.util.find_spec(pkg) is None:
        print('패키지 미설치:',pkg,'→ 설치 시도')
        res=subprocess.run([sys.executable,'-m','pip','install','-q',pkg])
        if res.returncode:
            print('⚠ 설치 실패:',pkg,'; 관련 기능이 제한될 수 있습니다')

os.environ['DS_FEATURE_MODE']=FEATURE_MODE
os.environ['TOKENIZERS_PARALLELISM']='false'
os.environ['PYTHONUNBUFFERED']='1'
WORK=Path('/kaggle/working');WORK.mkdir(exist_ok=True)
ROOT=WORK/'team_ensemble'/'D'
print('Configuration:',dict(PRESET=PRESET,FEATURE_MODE=FEATURE_MODE,DOWNLOAD_IMAGES=DOWNLOAD_IMAGES,MODEL_TOPN=MODEL_TOPN))

GPU: Tesla T4
Configuration: {'PRESET': 'MAX', 'FEATURE_MODE': 'auto', 'DOWNLOAD_IMAGES': True, 'MODEL_TOPN': 50}


In [2]:
# 원본 GitHub ZIP에 있는 D/src/*.py와 C의 teamkit.py 및 새 A 재구현 코드를 노트북 안에 내장
# 추가 GitHub 다운로드/비공개 저장소 인증 없이 실행 가능
import base64,io,zipfile,sys
SOURCE_ZIP_B64 = """..."""
ROOT.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(io.BytesIO(base64.b64decode(SOURCE_ZIP_B64))) as z:
    z.extractall(ROOT)
print('소스 설치:',len(list((ROOT/'src').glob('*.py'))),'개 파이썬 파일')
sys.path.insert(0,str(ROOT));import pipeline_helpers as P
ROOT=P.ROOT; SRC=P.SRC; OUT=P.OUT
OUT.mkdir(parents=True,exist_ok=True)

소스 설치: 12 개 파이썬 파일


In [3]:
# Kaggle Input 자동 탐색 + 공통 val/inner split 준비
import shutil,zipfile,json
INPUT=Path('/kaggle/input')
assert INPUT.exists(),'Kaggle Notebook 안에서 실행해야 합니다.'
train_candidates=[p for p in INPUT.rglob('train.csv') if (p.parent/'test.csv').exists() and 'validation' not in str(p).lower()]
if not train_candidates:
    # 학습 파일이 ZIP 내부에만 있는 경우 지원
    for zp in INPUT.rglob('*.zip'):
        try:
            with zipfile.ZipFile(zp) as z:
                names={Path(x).name:x for x in z.namelist()}
                if 'train.csv' in names and 'test.csv' in names:
                    tmp=WORK/'raw_competition';tmp.mkdir(exist_ok=True)
                    for name in ['train.csv','test.csv']:
                        (tmp/name).write_bytes(z.read(names[name]))
                    train_candidates=[tmp/'train.csv'];break
        except (OSError,zipfile.BadZipFile):pass
assert train_candidates,'train.csv + test.csv를 찾지 못했습니다. Kaggle Competition Data를 Add Input 해주세요.'
TRAIN=sorted(train_candidates,key=lambda p:('competition' not in str(p).lower(),len(str(p))))[0]
TEST=TRAIN.parent/'test.csv'
(ROOT/'data').mkdir(parents=True,exist_ok=True)
for source,dest in [(TRAIN,ROOT/'data/train.csv'),(TEST,ROOT/'data/test.csv')]:
    if not dest.exists():shutil.copy2(source,dest)

meta_candidates=list(INPUT.rglob('item_meta.jsonl'))
if not meta_candidates:
    for zp in INPUT.rglob('*.zip'):
        try:
            with zipfile.ZipFile(zp) as z:
                hit=next((x for x in z.namelist() if x.endswith('item_meta.jsonl')),None)
                if hit:
                    (ROOT/'data/item_meta.jsonl').write_bytes(z.read(hit));break
        except (OSError,zipfile.BadZipFile):pass
elif not (ROOT/'data/item_meta.jsonl').exists():
    shutil.copy2(meta_candidates[0],ROOT/'data/item_meta.jsonl')
HAS_META=(ROOT/'data/item_meta.jsonl').is_file()
print('Original train:',TRAIN,'| item_meta:',HAS_META)

VAL=ROOT/'teamkit/validation';VAL.mkdir(parents=True,exist_ok=True)
existing=[p for p in INPUT.rglob('manifest.json') if (p.parent/'val_train.csv').exists() and (p.parent/'val_truth.csv').exists() and (p.parent/'val_users.csv').exists()]
preferred=[]
for p in existing:
    try:
        if json.loads(p.read_text()).get('split_id')=='f0dfad2f4431a876':preferred.append(p)
    except Exception:pass
if preferred and not (VAL/'manifest.json').exists():
    for n in ['manifest.json','val_train.csv','val_truth.csv','val_users.csv']:
        shutil.copy2(preferred[0].parent/n,VAL/n)

sys.path.insert(0,str(ROOT));import teamkit
if not (VAL/'manifest.json').exists():
    manifest=teamkit.make_split(str(ROOT/'data/train.csv'),None,VAL,seed=42)
else:manifest=teamkit.verify_split(VAL)
print('VAL manifest:',manifest['split_id'],'| truth:',manifest.get('truth_rows'))
if manifest['split_id']!='f0dfad2f4431a876':
    raise ValueError('공통 Validation ID가 다릅니다. 반드시 팀 공통 데이터/원본 train.csv를 사용하세요.')
INNER=ROOT/'data/inner';INNER.mkdir(parents=True,exist_ok=True)
if not (INNER/'manifest.json').exists():
    inner_manifest=teamkit.make_split(str(VAL/'val_train.csv'),None,INNER,seed=7)
else:inner_manifest=teamkit.verify_split(INNER)
print('INNER manifest:',inner_manifest['split_id'],'| truth:',inner_manifest.get('truth_rows'))
assert inner_manifest['split_id']=='b45fa59b9ee68d8a','D파트와 동일한 내부 split이 아닙니다.'
for m in ['val','final','inner']:
    path=VAL/'val_users.csv' if m=='val' else ROOT/'data/test.csv' if m=='final' else INNER/'val_users.csv'
    print(m, 'users:',len(pd.read_csv(path)))

Original train: /kaggle/input/competitions/ssu-data-science-amazon-top-k-recommendation/train.csv | item_meta: True
VAL manifest: f0dfad2f4431a876 | truth: 137041
INNER manifest: b45fa59b9ee68d8a | truth: 109904
val users: 36043
final users: 36043
inner users: 36043


In [4]:
# 멀티모달 가용성 검사. 가능하면 사전학습 MiniLM/CLIP 사용; 불가능하면 proxy로 자동 전환.
import requests
images=ROOT/'data/images'
images.mkdir(parents=True,exist_ok=True)
if HAS_META and DOWNLOAD_IMAGES and FEATURE_MODE!='proxy':
    try:
        url=None
        with (ROOT/'data/item_meta.jsonl').open(encoding='utf-8') as f:
            for line in f:
                obj=json.loads(line)
                for im in (obj.get('images') or []):
                    url=next((im[k] for k in ['large','hi_res','thumb'] if im.get(k)),None)
                    if url:break
                if url:break
        if url:
            r=requests.get(url,timeout=6,headers={'User-Agent':'Mozilla/5.0'},stream=True)
            r.close()
            if r.status_code==200:
                print('이미지 서버 접속 가능 → 이미지 다운로드 (중단 시 기존 이미지 재사용)')
                P.run('download_images.py',['--workers','12'],skip=False,optional=True)
            else:print('이미지 서버 응답',r.status_code,'→ 이미지 proxy 사용 가능')
        else:print('item_meta에 이미지 URL 없음. 이미지 proxy 사용 가능')
    except Exception as ex:
        print('이미지 다운로드 생략:',repr(ex))
print('현재 이미지 파일:',len(list(images.glob('*.jpg'))))
# 서로 다른 실험 설정의 캐시를 실수로 혼합하지 않도록 방지
signature={'preset':PRESET,'feature_mode':FEATURE_MODE,'download_images':DOWNLOAD_IMAGES}
sigfile=ROOT/'pipeline_signature.json'
if sigfile.exists():
    old=json.loads(sigfile.read_text(encoding='utf8'))
    if old!=signature:
        raise RuntimeError(f'기존 모델 파일은 다른 설정으로 생성됐습니다: {old}. '
                           '새 Kaggle 세션을 사용하거나 /kaggle/working/team_ensemble 폴더를 정리하세요.')
else:sigfile.write_text(json.dumps(signature,indent=2),encoding='utf8')
if not HAS_META:print('⚠ item_meta.jsonl 부재. A text/meta, VBPR/FREEDOM 및 원래 D reranker는 재현할 수 없습니다.')

이미지 서버 접속 가능 → 이미지 다운로드 (중단 시 기존 이미지 재사용)
RUN: /usr/bin/python3 -u /kaggle/working/team_ensemble/D/src/download_images.py --workers 12
to download: 19543, no url: 4
1000/19543 done, failed 4
2000/19543 done, failed 4
3000/19543 done, failed 4
4000/19543 done, failed 4
5000/19543 done, failed 4
6000/19543 done, failed 4
7000/19543 done, failed 4
8000/19543 done, failed 4
9000/19543 done, failed 4
10000/19543 done, failed 4
11000/19543 done, failed 4
12000/19543 done, failed 4
13000/19543 done, failed 4
14000/19543 done, failed 4
15000/19543 done, failed 4
16000/19543 done, failed 4
17000/19543 done, failed 4
18000/19543 done, failed 4
19000/19543 done, failed 4
finished. failed/missing: 4
DONE (s): 50
현재 이미지 파일: 19543


## 1. A파트 원본 없는 모델 재구성 (A047_PROXY) — 모든 분할에서 별도 학습

A047 원본이 아니며, B 11-12의 BM25-Shrink + D의 TF-IDF 및 Metadata cosine을 혼합한다. 각 구성 채널과 하이브리드 **4개 모델 모두** `val / inner / final top100`을 만든다.

In [5]:
A_MODELS=['A_BM25','A_TEXT','A_META','A047_PROXY']
for mode in ['inner','val','final']:
    if not all(P.file_ok(n,mode) for n in A_MODELS):
        P.run('a_proxy.py',['--mode',mode]+(['--no-meta'] if not HAS_META else []),skip=False)
print('A 모델 완료:',{x:P.modes_ready(x) for x in A_MODELS})
if P.file_ok('A047_PROXY','val'):print('A047_PROXY',P.evaluate('A047_PROXY'))

RUN: /usr/bin/python3 -u /kaggle/working/team_ensemble/D/src/a_proxy.py --mode inner
A_PROXY mode=inner: users 36043, interactions 368997
 A_PROXY inner 8192/36043
 A_PROXY inner 16384/36043
 A_PROXY inner 24576/36043
 A_PROXY inner 32768/36043
 A_PROXY inner 36043/36043
saved A_BM25 inner
saved A_TEXT inner
saved A_META inner
saved A047_PROXY inner
DONE (s): 226
RUN: /usr/bin/python3 -u /kaggle/working/team_ensemble/D/src/a_proxy.py --mode val
A_PROXY mode=val: users 36043, interactions 478901
 A_PROXY val 8192/36043
 A_PROXY val 16384/36043
 A_PROXY val 24576/36043
 A_PROXY val 32768/36043
 A_PROXY val 36043/36043
saved A_BM25 val
saved A_TEXT val
saved A_META val
saved A047_PROXY val
DONE (s): 219
RUN: /usr/bin/python3 -u /kaggle/working/team_ensemble/D/src/a_proxy.py --mode final
A_PROXY mode=final: users 36043, interactions 615942
 A_PROXY final 8192/36043
 A_PROXY final 16384/36043
 A_PROXY final 24576/36043
 A_PROXY final 32768/36043
 A_PROXY final 36043/36043
saved A_BM25 final

## 2. D 콘텐츠 추천 (원본 코드 기반; 임베딩은 real 또는 proxy)

원본 D009가 MiniLM+CLIP 이미지+CLIP 텍스트를 사용한다. 이미지/모델이 없으면 메타데이터와 TF-IDF/SVD proxy로 대체한다. **이 경우 D009 원점수 재현이 아님**.

In [6]:
D_CONTENT='content_fuse_1_0.5_1_k10_pop7'
if HAS_META:
    for mode in ['inner','val','final']:
        P.run('run_content.py',['--features','fuse_1_0.5_1','--k','10','--pop-beta','7','--mode',mode,'--no-log'],
              expected=D_CONTENT,mode=mode,optional=ALLOW_OPTIONAL_FAILURE)
print('D_CONTENT 3 modes:',P.modes_ready(D_CONTENT))

RUN: /usr/bin/python3 -u /kaggle/working/team_ensemble/D/src/run_content.py --features fuse_1_0.5_1 --k 10 --pop-beta 7 --mode inner --no-log


Loading weights: 100%|██████████| 590/590 [00:00<00:00, 1419.26it/s]


features fuse_1_0.5_1: shape (19547, 1920), 329.5s
wrote /kaggle/working/team_ensemble/D/results/content_fuse_1_0.5_1_k10_pop7_inner.csv
DONE (s): 381
RUN: /usr/bin/python3 -u /kaggle/working/team_ensemble/D/src/run_content.py --features fuse_1_0.5_1 --k 10 --pop-beta 7 --mode val --no-log
features fuse_1_0.5_1: shape (19547, 1920), 1.6s
[content_fuse_1_0.5_1_k10_pop7] recall@20=0.04205 hits=5762 groups={'<=13': 0.04535, '14-18': 0.04353, '>=19': 0.0378}
DONE (s): 54
RUN: /usr/bin/python3 -u /kaggle/working/team_ensemble/D/src/run_content.py --features fuse_1_0.5_1 --k 10 --pop-beta 7 --mode final --no-log
features fuse_1_0.5_1: shape (19547, 1920), 1.5s
wrote /kaggle/working/team_ensemble/D/results/content_fuse_1_0.5_1_k10_pop7_final.csv
DONE (s): 50
D_CONTENT 3 modes: True


## 3. D VBPR (seed 0/1/2, 실험 preset에 따라 epoch 변경)

모든 seed를 `inner/val/final` 각각 학습한다. 3-seed 앙상블에서는 seed별 출력도 보존한다.

In [7]:
SEEDS=[0,1,2] if PRESET=='MAX' else [0]
VBPR_E=200 if PRESET=='MAX' else 100
VBPR_NAMES=[]
if HAS_META:
    for seed in SEEDS:
        name=f'vbpr_fuse_d64v64_lr0.001_reg0.005_e{VBPR_E}_s{seed}'
        for mode in ['inner','val','final']:
            P.run('vbpr.py',['--features','fuse','--d','64','--d-vis','64','--lr','0.001','--reg','0.005',
                           '--epochs',str(VBPR_E),'--eval-every',str(VBPR_E),'--seed',str(seed),'--mode',mode,'--no-log'],
                  expected=name,mode=mode,optional=ALLOW_OPTIONAL_FAILURE)
        if P.modes_ready(name):VBPR_NAMES.append(name)
print('VBPR seeds ready:',VBPR_NAMES)

RUN: /usr/bin/python3 -u /kaggle/working/team_ensemble/D/src/vbpr.py --features fuse --d 64 --d-vis 64 --lr 0.001 --reg 0.005 --epochs 200 --eval-every 200 --seed 0 --mode inner --no-log
epoch 1 loss 0.6705 (1.3s)
epoch 2 loss 0.5912 (0.8s)
epoch 3 loss 0.5372 (0.8s)
epoch 4 loss 0.4929 (0.8s)
epoch 5 loss 0.4505 (0.8s)
epoch 6 loss 0.4108 (0.8s)
epoch 7 loss 0.3750 (0.8s)
epoch 8 loss 0.3429 (0.8s)
epoch 9 loss 0.3146 (0.8s)
epoch 10 loss 0.2917 (0.8s)
epoch 11 loss 0.2712 (0.8s)
epoch 12 loss 0.2530 (0.8s)
epoch 13 loss 0.2388 (0.8s)
epoch 14 loss 0.2252 (0.8s)
epoch 15 loss 0.2137 (0.8s)
epoch 16 loss 0.2042 (0.8s)
epoch 17 loss 0.1958 (0.8s)
epoch 18 loss 0.1878 (0.8s)
epoch 19 loss 0.1800 (0.8s)
epoch 20 loss 0.1753 (0.8s)
epoch 21 loss 0.1698 (0.8s)
epoch 22 loss 0.1640 (0.8s)
epoch 23 loss 0.1602 (0.8s)
epoch 24 loss 0.1560 (0.8s)
epoch 25 loss 0.1525 (0.8s)
epoch 26 loss 0.1495 (0.8s)
epoch 27 loss 0.1466 (0.8s)
epoch 28 loss 0.1440 (0.8s)
epoch 29 loss 0.1411 (0.8s)
epoch 30 l

## 4. D FREEDOM (고정 멀티모달 그래프 + LightGCN; 실제 이미지 없으면 proxy modality)

원래 D031의 핵심 모델이다. MAX preset은 각 mode × 3 seeds × 100 epochs라 긴 실행이 필요하다.

In [8]:
FREEDOM_E=100 if PRESET=='MAX' else 75
FREEDOM_NAMES=[]
if HAS_META:
    for seed in SEEDS:
        name=f'freedom_d64_L2_k10_wi0.1_do0.8_rw0.0001_wd0_lr0.001_e{FREEDOM_E}_s{seed}'
        for mode in ['inner','val','final']:
            P.run('freedom.py',['--d','64','--n-layers','2','--n-mm-layers','1','--knn-k','10',
                         '--w-img','0.1','--dropout','0.8','--reg-weight','0.0001','--lr','0.001',
                         '--epochs',str(FREEDOM_E),'--eval-every',str(FREEDOM_E),'--seed',str(seed),
                         '--mode',mode,'--no-log'],
                  expected=name,mode=mode,optional=ALLOW_OPTIONAL_FAILURE)
        if P.modes_ready(name):FREEDOM_NAMES.append(name)
print('FREEDOM seeds ready:',FREEDOM_NAMES)

RUN: /usr/bin/python3 -u /kaggle/working/team_ensemble/D/src/freedom.py --d 64 --n-layers 2 --n-mm-layers 1 --knn-k 10 --w-img 0.1 --dropout 0.8 --reg-weight 0.0001 --lr 0.001 --epochs 100 --eval-every 100 --seed 0 --mode inner --no-log


/kaggle/working/team_ensemble/D/src/freedom.py:41: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
  return torch.sparse_coo_tensor(idx, v, (n, n)).coalesce()


epoch 1 loss 0.6902 (4.4s)
epoch 10 loss 0.3625 (4.1s)
epoch 20 loss 0.2496 (4.1s)
epoch 30 loss 0.1748 (4.1s)
epoch 40 loss 0.1261 (4.1s)
epoch 50 loss 0.0954 (4.1s)
epoch 60 loss 0.0759 (4.1s)
epoch 70 loss 0.0635 (4.1s)
epoch 80 loss 0.0534 (4.1s)
epoch 90 loss 0.0468 (4.1s)
epoch 100 loss 0.0420 (4.1s)
wrote /kaggle/working/team_ensemble/D/results/freedom_d64_L2_k10_wi0.1_do0.8_rw0.0001_wd0_lr0.001_e100_s0_inner.csv
DONE (s): 448
RUN: /usr/bin/python3 -u /kaggle/working/team_ensemble/D/src/freedom.py --d 64 --n-layers 2 --n-mm-layers 1 --knn-k 10 --w-img 0.1 --dropout 0.8 --reg-weight 0.0001 --lr 0.001 --epochs 100 --eval-every 100 --seed 0 --mode val --no-log


/kaggle/working/team_ensemble/D/src/freedom.py:41: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
  return torch.sparse_coo_tensor(idx, v, (n, n)).coalesce()


epoch 1 loss 0.6806 (5.7s)
epoch 10 loss 0.3625 (5.5s)
epoch 20 loss 0.2606 (5.5s)
epoch 30 loss 0.1845 (5.5s)
epoch 40 loss 0.1359 (5.5s)
epoch 50 loss 0.1065 (5.5s)
epoch 60 loss 0.0863 (5.5s)
epoch 70 loss 0.0721 (5.5s)
epoch 80 loss 0.0630 (5.5s)
epoch 90 loss 0.0556 (5.5s)
epoch 100 loss 0.0498 (5.5s)
[freedom_d64_L2_k10_wi0.1_do0.8_rw0.0001_wd0_lr0.001_e100_s0] recall@20=0.04705 hits=6448 groups={'<=13': 0.0486, '14-18': 0.0472, '>=19': 0.04572}
DONE (s): 596
RUN: /usr/bin/python3 -u /kaggle/working/team_ensemble/D/src/freedom.py --d 64 --n-layers 2 --n-mm-layers 1 --knn-k 10 --w-img 0.1 --dropout 0.8 --reg-weight 0.0001 --lr 0.001 --epochs 100 --eval-every 100 --seed 0 --mode final --no-log


/kaggle/working/team_ensemble/D/src/freedom.py:41: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
  return torch.sparse_coo_tensor(idx, v, (n, n)).coalesce()


... (9 outputs left out)

/kaggle/working/team_ensemble/D/src/freedom.py:41: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
  return torch.sparse_coo_tensor(idx, v, (n, n)).coalesce()


epoch 1 loss 0.6816 (5.7s)
epoch 10 loss 0.3587 (5.5s)
epoch 20 loss 0.2577 (5.5s)
epoch 30 loss 0.1847 (5.5s)
epoch 40 loss 0.1363 (5.5s)
epoch 50 loss 0.1058 (5.5s)
epoch 60 loss 0.0857 (5.5s)
epoch 70 loss 0.0716 (5.5s)
epoch 80 loss 0.0619 (5.5s)
epoch 90 loss 0.0544 (5.5s)
epoch 100 loss 0.0493 (5.5s)
[freedom_d64_L2_k10_wi0.1_do0.8_rw0.0001_wd0_lr0.001_e100_s2] recall@20=0.04618 hits=6328 groups={'<=13': 0.04745, '14-18': 0.04625, '>=19': 0.04514}
DONE (s): 596
RUN: /usr/bin/python3 -u /kaggle/working/team_ensemble/D/src/freedom.py --d 64 --n-layers 2 --n-mm-layers 1 --knn-k 10 --w-img 0.1 --dropout 0.8 --reg-weight 0.0001 --lr 0.001 --epochs 100 --eval-every 100 --seed 2 --mode final --no-log


/kaggle/working/team_ensemble/D/src/freedom.py:41: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
  return torch.sparse_coo_tensor(idx, v, (n, n)).coalesce()


epoch 1 loss 0.6607 (7.6s)
epoch 10 loss 0.3538 (7.4s)
epoch 20 loss 0.2626 (7.5s)
epoch 30 loss 0.1908 (7.5s)
epoch 40 loss 0.1438 (7.5s)
epoch 50 loss 0.1137 (7.5s)
epoch 60 loss 0.0945 (7.5s)
epoch 70 loss 0.0798 (7.5s)
epoch 80 loss 0.0713 (7.5s)
epoch 90 loss 0.0634 (7.5s)
epoch 100 loss 0.0578 (7.5s)
wrote /kaggle/working/team_ensemble/D/results/freedom_d64_L2_k10_wi0.1_do0.8_rw0.0001_wd0_lr0.001_e100_s2_final.csv
DONE (s): 787
FREEDOM seeds ready: ['freedom_d64_L2_k10_wi0.1_do0.8_rw0.0001_wd0_lr0.001_e100_s0', 'freedom_d64_L2_k10_wi0.1_do0.8_rw0.0001_wd0_lr0.001_e100_s1', 'freedom_d64_L2_k10_wi0.1_do0.8_rw0.0001_wd0_lr0.001_e100_s2']


## 5. D 협업 필터링: EASE / RP3β / iALS

모델별 val/inner/final Top100을 저장한다. iALS와 EASE는 GPU 사용량이 크므로 선택적으로 실패를 기록하고 건너뛴다.

In [9]:
D_CF={
'ease_l500': ['ease','--lam','500'],
'rp3b_a0.4_b0_k200':['rp3b','--alpha','0.4','--beta','0','--k','200'],
'ials_d128_a10_r100_it15_s0':['ials','--d','128','--alpha','10','--reg','100','--iters','15','--seed','0'],
}
for name,args in D_CF.items():
    for mode in ['inner','val','final']:
        P.run('cf.py',[*args,'--mode',mode,'--no-log'],expected=name,mode=mode,optional=ALLOW_OPTIONAL_FAILURE)
print('CF ready:',{n:P.modes_ready(n) for n in D_CF})

RUN: /usr/bin/python3 -u /kaggle/working/team_ensemble/D/src/cf.py ease --lam 500 --mode inner --no-log
wrote /kaggle/working/team_ensemble/D/results/ease_l500_inner.csv
DONE (s): 65
RUN: /usr/bin/python3 -u /kaggle/working/team_ensemble/D/src/cf.py ease --lam 500 --mode val --no-log
[ease_l500] recall@20=0.04345 hits=5955 groups={'<=13': 0.04201, '14-18': 0.04263, '>=19': 0.04552}
DONE (s): 69
RUN: /usr/bin/python3 -u /kaggle/working/team_ensemble/D/src/cf.py ease --lam 500 --mode final --no-log
wrote /kaggle/working/team_ensemble/D/results/ease_l500_final.csv
DONE (s): 66
RUN: /usr/bin/python3 -u /kaggle/working/team_ensemble/D/src/cf.py rp3b --alpha 0.4 --beta 0 --k 200 --mode inner --no-log
wrote /kaggle/working/team_ensemble/D/results/rp3b_a0.4_b0_k200_inner.csv
DONE (s): 58
RUN: /usr/bin/python3 -u /kaggle/working/team_ensemble/D/src/cf.py rp3b --alpha 0.4 --beta 0 --k 200 --mode val --no-log
[rp3b_a0.4_b0_k200] recall@20=0.04232 hits=5799 groups={'<=13': 0.0427, '14-18': 0.04179

CUDA runtime error: an illegal memory access was encountered (700) in magma_sgetrf_batched at /builder/magma/magma-cuda130/magma-2.6.1/src/sgetrf_batched.cpp:217
CUDA runtime error: an illegal memory access was encountered (700) in magma_sgetrf_batched at /builder/magma/magma-cuda130/magma-2.6.1/src/sgetrf_batched.cpp:218
CUDA runtime error: an illegal memory access was encountered (700) in magma_sgetrf_batched at /builder/magma/magma-cuda130/magma-2.6.1/src/sgetrf_batched.cpp:219
CUDA runtime error: an illegal memory access was encountered (700) in magma_queue_destroy_internal at /builder/magma/magma-cuda130/magma-2.6.1/interface_cuda/interface.cpp:1205
Traceback (most recent call last):
  File "/kaggle/working/team_ensemble/D/src/cf.py", line 135, in <module>
    main()
    ~~~~^^
  File "/kaggle/working/team_ensemble/D/src/cf.py", line 123, in main
    U, V = ials(X, a.d, a.alpha, a.reg, a.iters, a.seed)
           ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/workin

ERROR: cf.py exit code 1
RUN: /usr/bin/python3 -u /kaggle/working/team_ensemble/D/src/cf.py ials --d 128 --alpha 10 --reg 100 --iters 15 --seed 0 --mode val --no-log


CUDA runtime error: an illegal memory access was encountered (700) in magma_sgetrf_batched at /builder/magma/magma-cuda130/magma-2.6.1/src/sgetrf_batched.cpp:217
CUDA runtime error: an illegal memory access was encountered (700) in magma_sgetrf_batched at /builder/magma/magma-cuda130/magma-2.6.1/src/sgetrf_batched.cpp:218
CUDA runtime error: an illegal memory access was encountered (700) in magma_sgetrf_batched at /builder/magma/magma-cuda130/magma-2.6.1/src/sgetrf_batched.cpp:219
CUDA runtime error: an illegal memory access was encountered (700) in magma_queue_destroy_internal at /builder/magma/magma-cuda130/magma-2.6.1/interface_cuda/interface.cpp:1205
Traceback (most recent call last):
  File "/kaggle/working/team_ensemble/D/src/cf.py", line 135, in <module>
    main()
    ~~~~^^
  File "/kaggle/working/team_ensemble/D/src/cf.py", line 123, in main
    U, V = ials(X, a.d, a.alpha, a.reg, a.iters, a.seed)
           ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/workin

ERROR: cf.py exit code 1
RUN: /usr/bin/python3 -u /kaggle/working/team_ensemble/D/src/cf.py ials --d 128 --alpha 10 --reg 100 --iters 15 --seed 0 --mode final --no-log


CUDA runtime error: an illegal memory access was encountered (700) in magma_sgetrf_batched at /builder/magma/magma-cuda130/magma-2.6.1/src/sgetrf_batched.cpp:217
CUDA runtime error: an illegal memory access was encountered (700) in magma_sgetrf_batched at /builder/magma/magma-cuda130/magma-2.6.1/src/sgetrf_batched.cpp:218
CUDA runtime error: an illegal memory access was encountered (700) in magma_sgetrf_batched at /builder/magma/magma-cuda130/magma-2.6.1/src/sgetrf_batched.cpp:219
CUDA runtime error: an illegal memory access was encountered (700) in magma_queue_destroy_internal at /builder/magma/magma-cuda130/magma-2.6.1/interface_cuda/interface.cpp:1205
Traceback (most recent call last):
  File "/kaggle/working/team_ensemble/D/src/cf.py", line 135, in <module>
    main()
    ~~~~^^
  File "/kaggle/working/team_ensemble/D/src/cf.py", line 123, in main
    U, V = ials(X, a.d, a.alpha, a.reg, a.iters, a.seed)
           ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/workin

ERROR: cf.py exit code 1
CF ready: {'ease_l500': True, 'rp3b_a0.4_b0_k200': True, 'ials_d128_a10_r100_it15_s0': False}


## 6. Seed 앙상블 및 D 기반 종합 후보 생성 (Weighted RRF)

각 seed 파일은 유지하고 `D_VBPR_ENSEMBLE`, `D_FREEDOM_ENSEMBLE`, `D_BLEND6`의 inner/val/final CSV를 추가한다.

In [10]:
def seed_ensemble(names,alias):
    if not names:return None
    if len(names)==1:
        for mode in ['inner','val','final']:
            P.rename_output(names[0],alias,mode)
    else:
        for mode in ['inner','val','final']:
            P.run('blend.py',['--inputs',*names,'--name',alias,'--mode',mode,'--no-log'],
                  expected=alias,mode=mode,optional=ALLOW_OPTIONAL_FAILURE)
    return alias if P.modes_ready(alias) else None

V_ALIAS=seed_ensemble(VBPR_NAMES,'D_VBPR_ENSEMBLE')
F_ALIAS=seed_ensemble(FREEDOM_NAMES,'D_FREEDOM_ENSEMBLE')
print('seed ensemble:',V_ALIAS,F_ALIAS)

BASE_PARTS=[];BASE_WEIGHTS=[]
for name,w in [(D_CONTENT,1.0),(V_ALIAS,0.7),(F_ALIAS,1.3),('ease_l500',1.0),
               ('rp3b_a0.4_b0_k200',0.0),('ials_d128_a10_r100_it15_s0',0.0)]:
    if name and P.modes_ready(name) and w>0:
        BASE_PARTS.append(name);BASE_WEIGHTS.append(w)
if len(BASE_PARTS)<2:
    BASE_PARTS=['A047_PROXY','A_BM25'];BASE_WEIGHTS=[1.,0.3]
for mode in ['inner','val','final']:
    P.run('blend.py',['--inputs',*BASE_PARTS,'--weights',*map(str,BASE_WEIGHTS),
                      '--rrf-c','60','--name','D_BLEND6','--mode',mode,'--no-log'],
          expected='D_BLEND6',mode=mode,optional=False)
print('D_BLEND6',P.evaluate('D_BLEND6'))

RUN: /usr/bin/python3 -u /kaggle/working/team_ensemble/D/src/blend.py --inputs vbpr_fuse_d64v64_lr0.001_reg0.005_e200_s0 vbpr_fuse_d64v64_lr0.001_reg0.005_e200_s1 vbpr_fuse_d64v64_lr0.001_reg0.005_e200_s2 --name D_VBPR_ENSEMBLE --mode inner --no-log
wrote /kaggle/working/team_ensemble/D/results/D_VBPR_ENSEMBLE_inner.csv
DONE (s): 51
RUN: /usr/bin/python3 -u /kaggle/working/team_ensemble/D/src/blend.py --inputs vbpr_fuse_d64v64_lr0.001_reg0.005_e200_s0 vbpr_fuse_d64v64_lr0.001_reg0.005_e200_s1 vbpr_fuse_d64v64_lr0.001_reg0.005_e200_s2 --name D_VBPR_ENSEMBLE --mode val --no-log
[D_VBPR_ENSEMBLE] recall@20=0.03794 hits=5199 groups={'<=13': 0.03726, '14-18': 0.03553, '>=19': 0.04133}
DONE (s): 55
RUN: /usr/bin/python3 -u /kaggle/working/team_ensemble/D/src/blend.py --inputs vbpr_fuse_d64v64_lr0.001_reg0.005_e200_s0 vbpr_fuse_d64v64_lr0.001_reg0.005_e200_s1 vbpr_fuse_d64v64_lr0.001_reg0.005_e200_s2 --name D_VBPR_ENSEMBLE --mode final --no-log
wrote /kaggle/working/team_ensemble/D/results/D_

## 7. LightGBM binary reranker: inner로 학습, val로 평가, final로 예측

정답 누출 방지를 위해 inner에서만 학습한다. `TEAM_RERANK_inner_top100.csv.gz`도 생성하지만 **학습에 사용한 inner에 대한 in-sample 예측이므로 후속 스태킹 학습 입력으로 사용하면 안 된다.** Inner에서 OOF가 필요한 2차 모델을 만들려면 별도 교차 검증이 필요하다.

In [11]:
# 너무 유사한 LightGCN/SimpleX 계열은 D035 결과를 고려해 기본 제외
RANK_MODELS=['A047_PROXY','A_TEXT',D_CONTENT,V_ALIAS,F_ALIAS,'ease_l500',
             'rp3b_a0.4_b0_k200','ials_d128_a10_r100_it15_s0','D_BLEND6']
RANK_MODELS=list(dict.fromkeys(n for n in RANK_MODELS if n and P.modes_ready(n)))
print('RANK_MODELS:',RANK_MODELS)
RERANK_OK=False
if HAS_META and len(RANK_MODELS)>=3:
    RERANK_OK=P.run('rerank.py',[
       '--models',*RANK_MODELS,'--topn',str(MODEL_TOPN),
       '--apply','inner','val','final','--name','TEAM_RERANK',
       '--lr','0.02','--leaves','31','--min-leaf','500','--rounds','1200',
       '--objective','binary','--seeds','0','--no-log'],
       expected=None,skip=False,optional=ALLOW_OPTIONAL_FAILURE)
    RERANK_OK=RERANK_OK and P.modes_ready('TEAM_RERANK')
print('RERANK_OK:',RERANK_OK)
if RERANK_OK:print('TEAM_RERANK',P.evaluate('TEAM_RERANK'))

EXTRA_RERANK_OK=False
if RERANK_OK and RUN_EXTRA_RERANK:
    EXTRA_RERANK_OK=P.run('rerank.py',[
        '--models',*RANK_MODELS,'--topn',str(MODEL_TOPN),
        '--apply','inner','val','final','--name','TEAM_RERANK_EXTRA',
        '--lr','0.02','--leaves','31','--min-leaf','500','--rounds','1200',
        '--objective','binary','--seeds','0','1','2','--extra','--no-log'],
        expected=None,skip=False,optional=ALLOW_OPTIONAL_FAILURE)
    EXTRA_RERANK_OK=EXTRA_RERANK_OK and P.modes_ready('TEAM_RERANK_EXTRA')
print('EXTRA_RERANK_OK:',EXTRA_RERANK_OK)
if EXTRA_RERANK_OK:print('TEAM_RERANK_EXTRA',P.evaluate('TEAM_RERANK_EXTRA'))

RANK_MODELS: ['A047_PROXY', 'A_TEXT', 'content_fuse_1_0.5_1_k10_pop7', 'D_VBPR_ENSEMBLE', 'D_FREEDOM_ENSEMBLE', 'ease_l500', 'rp3b_a0.4_b0_k200', 'D_BLEND6']
RUN: /usr/bin/python3 -u /kaggle/working/team_ensemble/D/src/rerank.py --models A047_PROXY A_TEXT content_fuse_1_0.5_1_k10_pop7 D_VBPR_ENSEMBLE D_FREEDOM_ENSEMBLE ease_l500 rp3b_a0.4_b0_k200 D_BLEND6 --topn 50 --apply inner val final --name TEAM_RERANK --lr 0.02 --leaves 31 --min-leaf 500 --rounds 1200 --objective binary --seeds 0 --no-log


/kaggle/working/team_ensemble/D/src/rerank.py:152: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
  Xt = torch.sparse_csr_tensor(torch.as_tensor(X.indptr, dtype=torch.int64), torch.as_tensor(X.indices, dtype=torch.int64),
/kaggle/working/team_ensemble/D/src/rerank.py:152: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:49.)
  Xt = torch.sparse_csr_tensor(torch.as_tensor(X.indptr, dtype=torch.int64), torch.as_tensor(X.indices, dty

[inner] candidates 7,802,846 (216.5/user) in 29s
[inner] candidate recall (truth covered) 0.1691
Training until validation scores don't improve for 100 rounds
[100]	valid_0's ndcg@20: 0.155041
[200]	valid_0's ndcg@20: 0.155417
Early stopping, best iteration is:
[172]	valid_0's ndcg@20: 0.156323
[seed 0] best_iter 172 top features: r_mean=119346, rrf=44435, s_D_BLEND6=33316, timepop=18518, s_D_FREEDOM_ENSEMBLE=16015, user_n=15462, r_D_FREEDOM_ENSEMBLE=15033, dt_median=8172, m_leaf=7906, pop_share=6525, r_A047_PROXY=5736, item_first_minus_user_last=5179, s_A047_PROXY=5009, m_store=4668, s_D_VBPR_ENSEMBLE=4437
[inner] candidates 7,802,846 (216.5/user) in 27s
wrote /kaggle/working/team_ensemble/D/results/TEAM_RERANK_inner.csv
[val] candidates 7,514,302 (208.5/user) in 28s
[TEAM_RERANK] recall@20=0.06165 hits=8449 groups={'<=13': 0.06261, '14-18': 0.06158, '>=19': 0.06103}
[final] candidates 7,288,292 (202.2/user) in 27s
wrote /kaggle/working/team_ensemble/D/results/TEAM_RERANK_final.csv
DO

/kaggle/working/team_ensemble/D/src/rerank.py:152: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
  Xt = torch.sparse_csr_tensor(torch.as_tensor(X.indptr, dtype=torch.int64), torch.as_tensor(X.indices, dtype=torch.int64),
/kaggle/working/team_ensemble/D/src/rerank.py:152: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:49.)
  Xt = torch.sparse_csr_tensor(torch.as_tensor(X.indptr, dtype=torch.int64), torch.as_tensor(X.indices, dty

[inner] candidates 7,802,846 (216.5/user) in 31s
[inner] candidate recall (truth covered) 0.1691
Training until validation scores don't improve for 100 rounds
[100]	valid_0's ndcg@20: 0.15353
[200]	valid_0's ndcg@20: 0.154206
[300]	valid_0's ndcg@20: 0.155823
[400]	valid_0's ndcg@20: 0.156219
[500]	valid_0's ndcg@20: 0.155853
Early stopping, best iteration is:
[421]	valid_0's ndcg@20: 0.157039
[seed 0] best_iter 421 top features: r_mean=126547, s_D_BLEND6=38767, rrf=33143, timepop=24638, r_D_FREEDOM_ENSEMBLE=20343, user_n=16600, maxcos_mini=15401, s_D_FREEDOM_ENSEMBLE=13736, maxcos_clipimg=11815, dt_median=11717, item_first_minus_user_last=9985, pop_share=9840, r_A047_PROXY=8082, user_last_minus_item_last=8062, price=7997
Training until validation scores don't improve for 100 rounds
[100]	valid_0's ndcg@20: 0.167678
[200]	valid_0's ndcg@20: 0.170922
[300]	valid_0's ndcg@20: 0.171701
[400]	valid_0's ndcg@20: 0.173443
[500]	valid_0's ndcg@20: 0.172442
Early stopping, best iteration is:
[

## 8. Validation 기반 제한된 앙상블 탐색 → 최종 제출

가중치는 **공통 val_truth만 평가 용도로 사용**해 소수의 후보 중 선택한다. 모델 훈련에 val_truth를 넣지 않는다. 실제 Test 점수는 Kaggle에 제출한 뒤에만 알 수 있다.

In [12]:
FINAL_CANDIDATES=[]
for ranker in [n for n in ['TEAM_RERANK','TEAM_RERANK_EXTRA'] if P.modes_ready(n)]:
    FINAL_CANDIDATES.append((ranker,[ranker],[1.0]))
    # LightGBM + A047 재구현 및 D 앙상블의 보수적인 가중 RRF
    for a in [0.20,0.35,0.60]:
        FINAL_CANDIDATES.append((f'{ranker}_APROXY_{a:g}',
             [ranker,'A047_PROXY'],[1.0,a]))
    for d in [0.20,0.4]:
        FINAL_CANDIDATES.append((f'{ranker}_DBLEND_{d:g}',
             [ranker,'D_BLEND6'],[1.0,d]))
FINAL_CANDIDATES.extend([
  ('D_BLEND6',['D_BLEND6'],[1.0]),
  ('TEAM_A_D_RRF',['A047_PROXY','D_BLEND6'],[0.5,1.0]),
])

scores=[]
for name,names,weights in FINAL_CANDIDATES:
    if len(names)>1:
        for mode in ['inner','val','final']:
            P.run('blend.py',['--inputs',*names,'--weights',*map(str,weights),
                              '--rrf-c','60','--mode',mode,'--name',name,'--no-log'],
                  expected=name,mode=mode,optional=False)
    if P.modes_ready(name):
        r=P.evaluate(name,'val');r.update({'inputs':','.join(names),'weights':','.join(map(str,weights))});scores.append(r)

ranked=pd.DataFrame(scores).sort_values('recall@20',ascending=False)
ranked.to_csv(WORK/'team_candidate_ablation.csv',index=False)
print(ranked.to_string(index=False))
SELECTED=str(ranked.iloc[0]['model'])
assert P.modes_ready(SELECTED)
print('선택된 검증 최고 모델:',SELECTED,'val=',ranked.iloc[0]['recall@20'])
P.write_top20_submission(SELECTED,'final',WORK/'submission.csv')

RUN: /usr/bin/python3 -u /kaggle/working/team_ensemble/D/src/blend.py --inputs TEAM_RERANK A047_PROXY --weights 1.0 0.2 --rrf-c 60 --mode inner --name TEAM_RERANK_APROXY_0.2 --no-log
wrote /kaggle/working/team_ensemble/D/results/TEAM_RERANK_APROXY_0.2_inner.csv
DONE (s): 46
RUN: /usr/bin/python3 -u /kaggle/working/team_ensemble/D/src/blend.py --inputs TEAM_RERANK A047_PROXY --weights 1.0 0.2 --rrf-c 60 --mode val --name TEAM_RERANK_APROXY_0.2 --no-log
[TEAM_RERANK_APROXY_0.2] recall@20=0.06128 hits=8398 groups={'<=13': 0.06186, '14-18': 0.06117, '>=19': 0.06098}
DONE (s): 50
RUN: /usr/bin/python3 -u /kaggle/working/team_ensemble/D/src/blend.py --inputs TEAM_RERANK A047_PROXY --weights 1.0 0.2 --rrf-c 60 --mode final --name TEAM_RERANK_APROXY_0.2 --no-log
wrote /kaggle/working/team_ensemble/D/results/TEAM_RERANK_APROXY_0.2_final.csv
DONE (s): 45
RUN: /usr/bin/python3 -u /kaggle/working/team_ensemble/D/src/blend.py --inputs TEAM_RERANK A047_PROXY --weights 1.0 0.35 --rrf-c 60 --mode inne

PosixPath('/kaggle/working/submission.csv')

## 9. 품질 검사·공유용 CSV/ZIP·실험 환경 기록

모든 원천 모델에 **inner/val/final Top100**이 저장되었는지 점검한다. `TEAM_ALL_TOP100.zip` 안에는 전체 Top100 `.csv.gz`와 설정에 따라 압축 전 `.csv`도 포함된다.

In [13]:
import csv,hashlib
all_ready=sorted(set(p.name.split('_val_top100.csv.gz')[0] for p in OUT.glob('*_val_top100.csv.gz')
             if P.modes_ready(p.name.split('_val_top100.csv.gz')[0])))
print('모든 3모드 Top100 확보한 모델:',len(all_ready))
for name in all_ready:
    for mode in ['inner','val','final']:
        P.verify_top100(name,mode)
summary=P.summarize(WORK/'team_validation_scores.csv')
meta={
 'preset':PRESET,'feature_mode_requested':FEATURE_MODE,
 'feature_mode_note':'auto may mix real pretrained embeddings and TF-IDF/SVD proxies; use outputs as newly trained models, NOT exact D031.',
 'images_downloaded':len(list(images.glob('*.jpg'))),
 'embedding_backend_log':str(ROOT/'data/cache/embedding_backend_log.jsonl'),
 'has_item_meta':HAS_META,'validation_split':manifest['split_id'],
 'inner_split':inner_manifest['split_id'],'selected':SELECTED,
 'selected_val':float(ranked.iloc[0]['recall@20']),
 'ranker_training':'inner only; inner predictions of TEAM_RERANK are in-sample and must never be used for another stacking fit',
 'model_names_3modes':all_ready,'kaggle_public_score':'unknown until submitted',
 'ranker_features':RANK_MODELS,
}
(WORK/'team_run_config.json').write_text(json.dumps(meta,ensure_ascii=False,indent=2),encoding='utf-8')
archive=P.export_zip('TEAM_ALL_TOP100',plain_csv=INCLUDE_PLAIN_CSV_IN_ZIP)
print('OUTPUT FILES:')
for p in [WORK/'submission.csv',WORK/'team_run_config.json',WORK/'team_candidate_ablation.csv',
          WORK/'team_validation_scores.csv',WORK/'TEAM_ALL_TOP100.zip']:
    print(p,round(p.stat().st_size/1e6,2),'MB')

모든 3모드 Top100 확보한 모델: 29
Verified Top100 A047_PROXY inner rows 3604300
Verified Top100 A047_PROXY val rows 3604300
Verified Top100 A047_PROXY final rows 3604300
Verified Top100 A_BM25 inner rows 3604300
Verified Top100 A_BM25 val rows 3604300
Verified Top100 A_BM25 final rows 3604300
Verified Top100 A_META inner rows 3604300
Verified Top100 A_META val rows 3604300
Verified Top100 A_META final rows 3604300
Verified Top100 A_TEXT inner rows 3604300
Verified Top100 A_TEXT val rows 3604300
Verified Top100 A_TEXT final rows 3604300
Verified Top100 D_BLEND6 inner rows 3604300
Verified Top100 D_BLEND6 val rows 3604300
Verified Top100 D_BLEND6 final rows 3604300
Verified Top100 D_FREEDOM_ENSEMBLE inner rows 3604300
Verified Top100 D_FREEDOM_ENSEMBLE val rows 3604300
Verified Top100 D_FREEDOM_ENSEMBLE final rows 3604300
Verified Top100 D_VBPR_ENSEMBLE inner rows 3604300
Verified Top100 D_VBPR_ENSEMBLE val rows 3604300
Verified Top100 D_VBPR_ENSEMBLE final rows 3604300
Verified Top100 TEAM_A_D_R

### 학습 및 공유 시 유의 사항

- 모든 모델의 `inner`는 팀 공통 `val_train`을 seed 7로 분리해 학습한 결과다. `val`은 공통 `val_train`으로, `final`은 전체 `train.csv`로 각각 재학습한다.
- `A047_PROXY`의 실제 Validation 성능은 A047의 `0.052882`와 다를 수 있다. **같은 실험 ID를 사용하지 말고 새 ID**를 사용해 Notion에 기록한다.
- 이미지 파일 또는 Hugging Face 다운로드에 실패한 경우 proxy 임베딩으로 대체되므로 기존 D031 `0.06705` Public 점수 재현이 보장되지 않는다.
- `TEAM_RERANK_inner`는 inner 정답으로 학습한 모델의 **in-sample 예측**이다. 2차 앙상블 학습용 OOF 데이터가 아니다.
- `team_validation_scores.csv`에는 실제 실행으로 확인된 점수만 기록된다. `submission.csv`는 실제 Kaggle Test 예측이다.
- ZIP 파일이 매우 커질 수 있다. 결과가 사라지지 않게 `Save Version → Save & Run All`로 커밋 후 Output에서 다운로드한다.